# Score Fusion & Ensemble Analysis

Individual methods have varying strengths: Cosine Similarity and KS Test achieve above-random AUC,
while Wasserstein and Isolation Forest score below 0.5 (inversely correlated with anomalies).

This notebook investigates:
1. **Why Wasserstein and IF score below random** — score direction diagnosis
2. **Score fusion strategies** — combining methods to improve detection
3. **Rank-based ensemble** — method-agnostic combination via rank averaging
4. **Weighted ensemble** — optimised weights based on individual AUC performance

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from config import GENRES, genre_dataset_dir, genre_results_dir, RESULTS_DIR

OUT_DIR = RESULTS_DIR / 'ensemble_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {OUT_DIR}')
print(f'Genres     : {GENRES}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from scipy.stats import rankdata, spearmanr

In [ ]:
METHODS = {
    'Cosine Similarity':    'global_anomaly_score',
    'Wasserstein Distance': 'painting_swd_norm',
    'KS Test':              'ks_mean_d',
    'Isolation Forest':     'if_global_norm',
}

METHOD_COLORS = {
    'Cosine Similarity':    'steelblue',
    'Wasserstein Distance': 'darkorange',
    'KS Test':              'forestgreen',
    'Isolation Forest':     'crimson',
}

def load_genre_data(genre: str) -> pd.DataFrame:
    meta_path = genre_dataset_dir(genre, 'injected') / 'metadata.csv'
    r_dir     = genre_results_dir(genre, 'injected')
    meta   = pd.read_csv(meta_path)[['filename', 'is_anomaly', 'anomaly_genre']]
    cosine = pd.read_csv(r_dir / 'phase1_global_cosine_scores.csv')[['filename', 'global_anomaly_score']]
    swd    = pd.read_csv(r_dir / 'wasserstein_phase2_painting_scores.csv')[['filename', 'painting_swd_norm']]
    ks     = pd.read_csv(r_dir / 'ks_phase2_painting_scores.csv')[['filename', 'ks_mean_d']]
    if_    = pd.read_csv(r_dir / 'if_phase1_global_scores.csv')[['filename', 'if_global_norm']]
    df = meta.copy()
    for scores in [cosine, swd, ks, if_]:
        df = df.merge(scores, on='filename', how='left')
    df['genre'] = genre
    return df

genre_dfs = {g: load_genre_data(g) for g in GENRES}
all_data  = pd.concat(genre_dfs.values(), ignore_index=True)

for genre, df in genre_dfs.items():
    n = df['is_anomaly'].sum()
    print(f'{genre:<15} {len(df):>5} rows | anomalies: {n} ({n/len(df):.1%})')

## 1. Score Direction Diagnosis

AUC < 0.5 means a method assigns **lower** scores to true anomalies.
We check the correlation between each method's scores and the ground truth,
and test whether flipping the score direction improves detection.

In [ ]:
print('Raw AUC-ROC (before any correction):\n')
raw_aucs = {}
for genre, df in genre_dfs.items():
    y = df['is_anomaly'].values
    row = {}
    for method, col in METHODS.items():
        s = df[col].fillna(df[col].median()).values
        row[method] = roc_auc_score(y, s)
    raw_aucs[genre] = row
raw_df = pd.DataFrame(raw_aucs).T
raw_df.loc['Mean'] = raw_df.mean()
print(raw_df.round(4).to_string())

print('\n\nCorrected AUC-ROC (flip scores where AUC < 0.5):\n')
corrected_aucs = {}
for genre, df in genre_dfs.items():
    y = df['is_anomaly'].values
    row = {}
    for method, col in METHODS.items():
        s = df[col].fillna(df[col].median()).values
        auc = roc_auc_score(y, s)
        # If AUC < 0.5, the score is inversely related to anomalousness
        if auc < 0.5:
            auc = roc_auc_score(y, -s)
            row[method] = auc
        else:
            row[method] = auc
    corrected_aucs[genre] = row
corr_df = pd.DataFrame(corrected_aucs).T
corr_df.loc['Mean'] = corr_df.mean()
print(corr_df.round(4).to_string())

print('\n\nInterpretation:')
for method in METHODS:
    raw_mean = raw_df.loc['Mean', method]
    cor_mean = corr_df.loc['Mean', method]
    if raw_mean < 0.5:
        print(f'  {method}: raw AUC={raw_mean:.3f} → flipped AUC={cor_mean:.3f}')
        print(f'    → Score is INVERSELY correlated with anomalousness.')
        print(f'    → These methods assign LOWER scores to anomalies (they look normal to the detector).')
    else:
        print(f'  {method}: AUC={raw_mean:.3f} — score direction is correct.')

## 2. Score Correlation Between Methods

How much do the four methods agree? High Spearman correlation suggests
redundant information; low correlation means ensemble fusion may help.

In [ ]:
fig, axes = plt.subplots(1, len(GENRES), figsize=(6 * len(GENRES), 5))

for ax, genre in zip(axes, GENRES):
    df = genre_dfs[genre]
    score_cols = list(METHODS.values())
    corr_matrix = df[score_cols].corr(method='spearman')
    corr_matrix.index = list(METHODS.keys())
    corr_matrix.columns = list(METHODS.keys())

    im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(METHODS)))
    ax.set_yticks(range(len(METHODS)))
    ax.set_xticklabels(list(METHODS.keys()), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(list(METHODS.keys()), fontsize=8)
    for i in range(len(METHODS)):
        for j in range(len(METHODS)):
            val = corr_matrix.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(val) > 0.6 else 'black')
    ax.set_title(genre.title(), fontsize=11)

plt.colorbar(im, ax=axes[-1], label='Spearman ρ', shrink=0.8)
fig.suptitle('Score Correlation Between Methods (Spearman)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'method_correlation_heatmap.png', dpi=150)
plt.show()

## 3. Ensemble Strategies

We test several fusion approaches:

| Strategy | Description |
|---|---|
| **Mean Score** | Average normalised scores from all 4 methods |
| **Top-2 Mean** | Average of Cosine + KS (the two best individual methods) |
| **Rank Fusion** | Convert each method's scores to ranks, then average ranks |
| **AUC-Weighted** | Weight methods by their individual AUC (per genre) |
| **Corrected Mean** | Flip below-random methods, then average all 4 |

In [ ]:
def normalise_01(x):
    """Min-max normalise to [0, 1]."""
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


ensemble_records = []

for genre, df in genre_dfs.items():
    y = df['is_anomaly'].values
    scores = {}
    for method, col in METHODS.items():
        s = df[col].fillna(df[col].median()).values
        scores[method] = normalise_01(s)

    # Individual AUCs for weighting
    indiv_auc = {m: roc_auc_score(y, s) for m, s in scores.items()}

    # --- Strategy 1: Mean of all 4 ---
    mean_all = np.mean(list(scores.values()), axis=0)

    # --- Strategy 2: Top-2 (Cosine + KS) ---
    top2 = (scores['Cosine Similarity'] + scores['KS Test']) / 2

    # --- Strategy 3: Rank fusion ---
    ranks = np.mean([rankdata(s) for s in scores.values()], axis=0)

    # --- Strategy 4: AUC-weighted ---
    weights = np.array([max(indiv_auc[m], 1 - indiv_auc[m]) for m in METHODS])  # flip below-random
    weights = weights / weights.sum()
    weighted = np.zeros(len(y))
    for w, (method, _) in zip(weights, METHODS.items()):
        s = scores[method]
        # Flip score direction for below-random methods
        if indiv_auc[method] < 0.5:
            s = 1 - s
        weighted += w * s

    # --- Strategy 5: Corrected mean (flip below-random) ---
    corrected = []
    for method in METHODS:
        s = scores[method]
        if indiv_auc[method] < 0.5:
            s = 1 - s
        corrected.append(s)
    corrected_mean = np.mean(corrected, axis=0)

    # Evaluate all strategies
    strategies = {
        'Mean (all 4)':       mean_all,
        'Top-2 (Cos+KS)':    top2,
        'Rank Fusion':        ranks,
        'AUC-Weighted':       weighted,
        'Corrected Mean':     corrected_mean,
    }

    # Also include individual methods for comparison
    for method in METHODS:
        strategies[method] = scores[method]

    for name, s in strategies.items():
        auc = roc_auc_score(y, s)
        ap  = average_precision_score(y, s)
        ensemble_records.append({
            'genre': genre, 'strategy': name,
            'AUC-ROC': auc, 'Avg Precision': ap
        })

ens_df = pd.DataFrame(ensemble_records)
ens_pivot = ens_df.pivot(index='strategy', columns='genre', values='AUC-ROC').round(4)
ens_pivot['Mean AUC'] = ens_pivot.mean(axis=1)
ens_pivot = ens_pivot.sort_values('Mean AUC', ascending=False)

print('Ensemble vs Individual — AUC-ROC:\n')
print(ens_pivot.to_string())

In [ ]:
# Highlight ensemble improvements over best individual method
ensemble_names = ['Mean (all 4)', 'Top-2 (Cos+KS)', 'Rank Fusion',
                  'AUC-Weighted', 'Corrected Mean']
individual_names = list(METHODS.keys())

ens_only  = ens_pivot.loc[ens_pivot.index.isin(ensemble_names)]
ind_only  = ens_pivot.loc[ens_pivot.index.isin(individual_names)]
best_ind  = ind_only['Mean AUC'].max()
best_ind_name = ind_only['Mean AUC'].idxmax()

print(f'Best individual method: {best_ind_name} (Mean AUC = {best_ind:.4f})\n')
print('Ensemble strategies vs best individual:\n')
for name, row in ens_only.iterrows():
    delta = row['Mean AUC'] - best_ind
    arrow = '↑' if delta > 0 else '↓'
    print(f'  {name:<25} Mean AUC = {row["Mean AUC"]:.4f}  ({arrow} {abs(delta):.4f})')

In [ ]:
# ROC curves: individual vs best ensembles
fig, axes = plt.subplots(1, len(GENRES), figsize=(6 * len(GENRES), 5), sharey=True)

ensemble_colors = {
    'Top-2 (Cos+KS)':  'purple',
    'AUC-Weighted':     'darkgoldenrod',
    'Corrected Mean':   'teal',
}

for ax, genre in zip(axes, GENRES):
    df = genre_dfs[genre]
    y  = df['is_anomaly'].values

    # Individual methods (thin lines)
    for method, col in METHODS.items():
        s = df[col].fillna(df[col].median()).values
        fpr, tpr, _ = roc_curve(y, s)
        auc = roc_auc_score(y, s)
        ax.plot(fpr, tpr, color=METHOD_COLORS[method], linewidth=1, alpha=0.4,
                label=f'{method} ({auc:.3f})')

    # Ensemble methods (thick lines)
    scores = {m: normalise_01(df[c].fillna(df[c].median()).values) for m, c in METHODS.items()}
    indiv_auc = {m: roc_auc_score(y, s) for m, s in scores.items()}

    top2 = (scores['Cosine Similarity'] + scores['KS Test']) / 2

    weights = np.array([max(indiv_auc[m], 1 - indiv_auc[m]) for m in METHODS])
    weights = weights / weights.sum()
    weighted = np.zeros(len(y))
    for w, method in zip(weights, METHODS):
        s = scores[method]
        if indiv_auc[method] < 0.5:
            s = 1 - s
        weighted += w * s

    corrected = []
    for method in METHODS:
        s = scores[method]
        if indiv_auc[method] < 0.5:
            s = 1 - s
        corrected.append(s)
    corrected_mean = np.mean(corrected, axis=0)

    ensembles = {
        'Top-2 (Cos+KS)': top2,
        'AUC-Weighted':    weighted,
        'Corrected Mean':  corrected_mean,
    }

    for ename, es in ensembles.items():
        fpr, tpr, _ = roc_curve(y, es)
        auc = roc_auc_score(y, es)
        ax.plot(fpr, tpr, color=ensemble_colors[ename], linewidth=2.5,
                label=f'{ename} ({auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.4)
    ax.set_title(genre.title(), fontsize=12)
    ax.set_xlabel('False Positive Rate')
    if ax is axes[0]:
        ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)

fig.suptitle('ROC Curves — Individual Methods vs Ensemble Strategies',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'roc_ensemble_vs_individual.png', dpi=150)
plt.show()

## 4. Per-Anomaly-Genre Breakdown

The injected anomalies come from 3 source genres (Cubism, Expressionism, Abstract Expressionism).
Are some source genres easier to detect than others? This reveals which
art styles are most/least distinguishable from the target genre.

In [ ]:
anomaly_records = []

for genre, df in genre_dfs.items():
    anomalies = df[df['is_anomaly'] == 1]
    normals   = df[df['is_anomaly'] == 0]

    for src_genre in anomalies['anomaly_genre'].unique():
        # Create a subset: all normals + anomalies from this source genre only
        src_mask = anomalies['anomaly_genre'] == src_genre
        sub = pd.concat([normals, anomalies[src_mask]], ignore_index=True)
        y_sub = sub['is_anomaly'].values
        n_anom = y_sub.sum()

        for method, col in METHODS.items():
            s = sub[col].fillna(sub[col].median()).values
            auc = roc_auc_score(y_sub, s)
            anomaly_records.append({
                'target_genre': genre,
                'source_genre': src_genre,
                'method': method,
                'AUC-ROC': auc,
                'n_anomalies': int(n_anom),
            })

anomaly_df = pd.DataFrame(anomaly_records)

# Pivot: which source genres are easiest to detect?
for genre in GENRES:
    sub = anomaly_df[anomaly_df['target_genre'] == genre]
    piv = sub.pivot(index='method', columns='source_genre', values='AUC-ROC').round(4)
    piv['Mean'] = piv.mean(axis=1)
    piv = piv.sort_values('Mean', ascending=False)
    print(f'\n=== {genre.title()} — AUC by Anomaly Source Genre ===\n')
    print(piv.to_string())

In [ ]:
# Heatmap: detectability by source genre x target genre (best method only)
best_method = 'Cosine Similarity'
sub = anomaly_df[anomaly_df['method'] == best_method]
piv = sub.pivot(index='target_genre', columns='source_genre', values='AUC-ROC')

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(piv.values, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='AUC-ROC')
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns, fontsize=10)
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels([g.title() for g in piv.index], fontsize=10)
for i in range(len(piv.index)):
    for j in range(len(piv.columns)):
        val = piv.values[i, j]
        color = 'white' if val < 0.55 or val > 0.9 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color=color)
ax.set_xlabel('Anomaly Source Genre')
ax.set_ylabel('Target Genre')
ax.set_title(f'Anomaly Detectability by Source Genre ({best_method})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'detectability_by_source_genre.png', dpi=150)
plt.show()

## 5. Bootstrap Confidence Intervals

AUC point estimates are not enough for a paper — we need confidence intervals
to determine whether differences between methods are statistically significant.

In [ ]:
def bootstrap_auc(y_true, scores, n_boot=2000, seed=42):
    """Compute bootstrap 95% CI for AUC-ROC."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    aucs = np.zeros(n_boot)
    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        # Ensure both classes are present in the bootstrap sample
        if len(np.unique(y_true[idx])) < 2:
            aucs[i] = np.nan
            continue
        aucs[i] = roc_auc_score(y_true[idx], scores[idx])
    aucs = aucs[~np.isnan(aucs)]
    return np.percentile(aucs, [2.5, 97.5]), aucs


ci_records = []

for genre, df in genre_dfs.items():
    y = df['is_anomaly'].values
    for method, col in METHODS.items():
        s = df[col].fillna(df[col].median()).values
        auc = roc_auc_score(y, s)
        (lo, hi), _ = bootstrap_auc(y, s)
        ci_records.append({
            'genre': genre, 'method': method,
            'AUC': auc, 'CI_lo': lo, 'CI_hi': hi,
        })

    # Also add best ensemble
    scores = {m: normalise_01(df[c].fillna(df[c].median()).values) for m, c in METHODS.items()}
    top2 = (scores['Cosine Similarity'] + scores['KS Test']) / 2
    auc = roc_auc_score(y, top2)
    (lo, hi), _ = bootstrap_auc(y, top2)
    ci_records.append({
        'genre': genre, 'method': 'Ensemble (Cos+KS)',
        'AUC': auc, 'CI_lo': lo, 'CI_hi': hi,
    })

ci_df = pd.DataFrame(ci_records)
ci_df['CI'] = ci_df.apply(lambda r: f"[{r['CI_lo']:.3f}, {r['CI_hi']:.3f}]", axis=1)

for genre in GENRES:
    sub = ci_df[ci_df['genre'] == genre].sort_values('AUC', ascending=False)
    print(f'\n=== {genre.title()} ===\n')
    print(sub[['method', 'AUC', 'CI']].to_string(index=False))

In [ ]:
# Forest plot: AUC with 95% CI
fig, axes = plt.subplots(1, len(GENRES), figsize=(6 * len(GENRES), 5), sharey=True)

all_methods_ordered = ['Ensemble (Cos+KS)', 'Cosine Similarity', 'KS Test',
                       'Wasserstein Distance', 'Isolation Forest']
ci_colors = {'Ensemble (Cos+KS)': 'purple', **METHOD_COLORS}

for ax, genre in zip(axes, GENRES):
    sub = ci_df[ci_df['genre'] == genre].set_index('method')
    for i, method in enumerate(all_methods_ordered):
        if method not in sub.index:
            continue
        row = sub.loc[method]
        color = ci_colors.get(method, 'gray')
        ax.errorbar(row['AUC'], i, xerr=[[row['AUC'] - row['CI_lo']],
                    [row['CI_hi'] - row['AUC']]], fmt='o', color=color,
                    capsize=5, capthick=2, markersize=8, linewidth=2)

    ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_yticks(range(len(all_methods_ordered)))
    if ax is axes[0]:
        ax.set_yticklabels(all_methods_ordered, fontsize=9)
    ax.set_xlabel('AUC-ROC')
    ax.set_title(genre.title(), fontsize=12)
    ax.set_xlim(0.2, 1.0)
    ax.grid(alpha=0.3, axis='x')

fig.suptitle('AUC-ROC with 95% Bootstrap Confidence Intervals',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'auc_confidence_intervals.png', dpi=150)
plt.show()

## 6. Statistical Significance: Paired Bootstrap Test

Test whether the best method significantly outperforms the runner-up
using a paired bootstrap test on AUC differences.

In [ ]:
def paired_bootstrap_test(y_true, scores_a, scores_b, n_boot=5000, seed=42):
    """Test H0: AUC(A) = AUC(B). Returns p-value (two-sided)."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    observed_diff = roc_auc_score(y_true, scores_a) - roc_auc_score(y_true, scores_b)
    diffs = np.zeros(n_boot)
    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            diffs[i] = 0
            continue
        diffs[i] = (roc_auc_score(y_true[idx], scores_a[idx]) -
                    roc_auc_score(y_true[idx], scores_b[idx]))
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))
    return observed_diff, p_value


print('Pairwise significance tests (Cosine vs each other method):\n')
for genre, df in genre_dfs.items():
    y = df['is_anomaly'].values
    ref_scores = df['global_anomaly_score'].fillna(df['global_anomaly_score'].median()).values
    print(f'--- {genre.title()} ---')
    for method, col in METHODS.items():
        if method == 'Cosine Similarity':
            continue
        other_scores = df[col].fillna(df[col].median()).values
        diff, p = paired_bootstrap_test(y, ref_scores, other_scores)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  Cosine vs {method:<25} ΔAUC = {diff:+.4f}  p = {p:.4f} {sig}')
    print()

## 7. Summary

Save complete ensemble results for the paper.

In [ ]:
# Save all results
ens_df.to_csv(OUT_DIR / 'ensemble_results.csv', index=False)
ci_df.to_csv(OUT_DIR / 'bootstrap_confidence_intervals.csv', index=False)
anomaly_df.to_csv(OUT_DIR / 'per_source_genre_detectability.csv', index=False)

print('Saved:')
print(f'  {OUT_DIR / "ensemble_results.csv"}')
print(f'  {OUT_DIR / "bootstrap_confidence_intervals.csv"}')
print(f'  {OUT_DIR / "per_source_genre_detectability.csv"}')
print()
print('Key findings for the paper:')
print('  1. Score direction diagnosis — why Wasserstein/IF underperform')
print('  2. Score correlation — method independence analysis')
print('  3. Ensemble fusion — does combining methods improve detection?')
print('  4. Per-source-genre analysis — which anomaly types are hardest?')
print('  5. Bootstrap CIs — statistical rigor for AUC comparisons')
print('  6. Significance tests — are method differences real?')